# Step 16.2 — Train / Validation Split：为什么要把一部分 Tokens 永远留在训练之外？

Step 16.1 已经得到一条完整 character-level token stream：

$$
token\_stream:(N,).
$$

现在的问题是：

> 如果模型训练时看过整份 corpus，我们以后怎样判断它是在学习可泛化的模式，还是只是在越来越适应训练数据本身？

因此本节建立一个最基本的数据边界：

```text
full token stream (N,)
        ↓ contiguous split
train tokens (N_train,)
validation tokens (N_val,)
```

本节只做 **held-out train/validation split**。

不进入 Step 16.3：不定义 `block_size`、不采样 windows、不生成 `(B,T)` batches。


## 0. 新 Work Copy 怎样取得 Step 16.1 的 `token_stream`？

16.2 依赖 16.1 已经缓存的：

```text
data/tinyshakespeare/input.txt
```

但不同 Notebook 不应假设共享同一个 Jupyter kernel。

所以 16.2 母版提供一个最小 **local-cache bootstrap helper**：

```text
cached UTF-8 file
↓
character vocabulary
↓
stoi / itos
↓
full encoded MLX token_stream (N,)
```

它只读取本地 cache，**不会联网下载**。

如果 cache 不存在，会明确报错并提示：

> 回到 Step 16.1 work copy，执行 acquisition usage cell 一次。

因此 16.2 不需要 `%run` 整份 16.1 Notebook，也不会因为“加载依赖”而顺带执行 16.1 的 preview / assertions。


In [ ]:
from pathlib import Path
import mlx.core as mx


CACHE_PATH = Path('data/tinyshakespeare/input.txt')


def load_cached_character_corpus(cache_path: Path = CACHE_PATH):
    """Load the Step 16.1 local cache and rebuild its character token stream."""

    cache_path = Path(cache_path)

    if not cache_path.exists():
        raise FileNotFoundError(
            'Tiny Shakespeare cache not found at '
            f'{cache_path}. Complete the Step 16.1 acquisition usage cell first.'
        )

    text = cache_path.read_text(encoding='utf-8')
    characters = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(characters)}
    itos = {i: ch for i, ch in enumerate(characters)}

    encoded_ids = [stoi[ch] for ch in text]
    token_stream = mx.array(encoded_ids, dtype=mx.int32)

    return text, characters, stoi, itos, token_stream


# Definition only: no file is read and no network is accessed here.
# The master notebook remains unexecuted.


In [ ]:
# USER-RUN LOCAL-BOOTSTRAP CELL — prepared only, NOT executed in the master.

text, characters, stoi, itos, token_stream = load_cached_character_corpus()

print('corpus characters:', len(text))
print('vocab size:', len(characters))
print('token_stream shape:', token_stream.shape)
print('token_stream dtype:', token_stream.dtype)

# Expected contract from Step 16.1:
# token_stream is a 1-D integer MLX array with shape (N,).
# This cell does NOT download anything.


## 1. Held-Out Validation 到底是为了什么？

训练数据（training data）的作用是：

> 给 optimizer 提供允许改变模型参数的学习信号。

验证数据（validation data）的作用不同：

> 在模型**没有用这些 token 做参数更新**的前提下，检查模型对 held-out text 的表现。

因此最核心的规则是：

$$
\boxed{
\text{validation tokens must not be used for model parameter updates}
}.
$$

以后训练时可以周期性地在 validation split 上计算同一种评估量 / loss，但：

```text
train split
-> forward
-> loss
-> gradients
-> optimizer update

validation split
-> forward
-> evaluation
-> NO parameter update
```

本节还不实现 loss、gradient 或 optimizer；这里只先把数据边界建立正确。


## 2. 为什么这门课先用 Contiguous Split？

对于一条连续文本 token stream：

$$
[x_0,x_1,\ldots,x_{N-1}],
$$

我们采用最简单的连续切分：

```text
earlier 90%                  final 10%
┌──────────────────────────┬─────────┐
│       training           │   val   │
└──────────────────────────┴─────────┘
0                      split_idx      N
```

定义：

$$
r=0.9
$$

$$
\boxed{
N_{train}=\lfloor rN\rfloor
}
$$

$$
\boxed{
N_{val}=N-N_{train}
}.
$$

然后：

$$
train=token\_stream[:N_{train}],
$$

$$
val=token\_stream[N_{train}:].
$$

两个 slice 的原始 index 范围不重叠，因此 validation token positions 不会同时属于 training split。


## 3. Tiny Example：20 个 Tokens 怎样切？

假设：

$$
N=20
$$

并使用：

$$
r=0.9.
$$

则：

$$
N_{train}
=
\lfloor0.9\times20\rfloor
=
18.
$$

$$
N_{val}
=
20-18
=
2.
$$

原 token positions：

```text
0  1  2  ...  16 17 | 18 19
└────── train ───────┘ └ val ┘
```

得到：

$$
train.shape=(18,),
$$

$$
val.shape=(2,).
$$

这里只切**数据 ownership**；还没有从任何一侧取固定长度 training window。


## 4. 把 Split Contract 写成一个小 Helper

我们希望 helper 明确检查：

- 输入必须是 1-D token stream；
- corpus 至少有两个 tokens；
- `train_fraction` 必须满足 `0 < r < 1`；
- split 后 train / validation 都非空。

输出：

```text
train_tokens : (N_train,)
val_tokens   : (N_val,)
split_index  : Python int
```

这里不 shuffle，也不构造随机 permutation。

随机窗口采样属于 Step 16.3。


In [ ]:
def split_train_validation(
    token_stream: mx.array,
    train_fraction: float = 0.90,
):
    """Contiguously split a 1-D token stream into held-out train/validation data."""

    if len(token_stream.shape) != 1:
        raise ValueError('token_stream must have shape (N,)')

    N = token_stream.shape[0]

    if N < 2:
        raise ValueError('token_stream must contain at least 2 tokens')
    if not 0.0 < train_fraction < 1.0:
        raise ValueError('train_fraction must satisfy 0 < train_fraction < 1')

    split_index = int(N * train_fraction)

    if split_index <= 0 or split_index >= N:
        raise ValueError('split would make train or validation data empty')

    train_tokens = token_stream[:split_index]
    val_tokens = token_stream[split_index:]

    return train_tokens, val_tokens, split_index


# Definition only; no split has been performed in the master notebook.


In [ ]:
# USER-RUN SPLIT / INSPECTION CELL — prepared only, NOT executed in the master.

train_tokens, val_tokens, split_index = split_train_validation(
    token_stream,
    train_fraction=0.90,
)

N = token_stream.shape[0]
N_train = train_tokens.shape[0]
N_val = val_tokens.shape[0]

assert N_train + N_val == N
assert N_train == split_index
assert N_val == N - split_index

print('full token stream:', token_stream.shape)
print('split index:', split_index)
print('train tokens:', train_tokens.shape)
print('validation tokens:', val_tokens.shape)
print('train fraction (actual):', N_train / N)
print('validation fraction (actual):', N_val / N)

# No split or assertion has run in the master notebook.


## 5. “Held-Out” 的真正含义：不是文件名不同，而是训练索引不重叠

我们切分的是原始 corpus token positions。

Train index set：

$$
\mathcal I_{train}
=
\{0,\ldots,N_{train}-1\}.
$$

Validation index set：

$$
\mathcal I_{val}
=
\{N_{train},\ldots,N-1\}.
$$

因此：

$$
\boxed{
\mathcal I_{train}\cap\mathcal I_{val}=\varnothing
}.
$$

这才是当前 held-out split 的关键。

以后 Step 16.3 构造 windows 时也必须遵守这个边界：

```text
training window
-> 完全来自 train_tokens

validation window
-> 完全来自 val_tokens
```

不能让一个 training window 跨过 `split_index` 偷吃 validation token。

本节只解释这个约束，不实现 window / `block_size`。


## 6. 一个容易混淆的边界：Vocabulary 与 Held-Out Sequence Tokens

Step 16.1 已经从完整 Tiny Shakespeare corpus 建立：

```text
characters = sorted(set(text))
stoi / itos
```

Step 16.2 沿用这个既有 tokenizer contract，不重新设计 tokenizer。

因此这里的 held-out 规则具体指：

> **validation token sequence 不参与模型参数更新。**

它不是说我们这一节要重新训练一个“只看 train split 的 tokenizer”。

对于当前教学用 fixed character vocabulary，这个选择让 train / validation 两边共享同一组 token IDs：

$$
0,1,\ldots,V-1.
$$

以后模型的 embedding vocabulary size V 对两边完全一致。

更复杂的数据工程中，tokenizer construction 本身也可以有独立的数据治理规则；本课程当前不扩展到那里。


## 7. Split 之后 Shape 到底是什么？

16.1：

$$
token\_stream:(N,).
$$

16.2：

$$
train\_tokens:(N_{train},),
$$

$$
val\_tokens:(N_{val},).
$$

并满足：

$$
\boxed{
N_{train}+N_{val}=N
}.
$$

注意它们仍然全部是 **1-D token streams**。

现在还没有：

$$
(B,T).
$$

`B` 与固定 context/window length 会在 Step 16.3 batch sampler 才出现。


## 8. 为什么不把单个 Tokens 随机撒到 Train / Validation？

一个错误思路是：

```text
token 0 -> train
token 1 -> validation
token 2 -> train
token 3 -> validation
...
```

这样会破坏原始连续文本结构。

Autoregressive LM 需要学习：

$$
p(x_{t+1}\mid x_{\le t}),
$$

而上下文来自连续 token sequence。

所以当前教学 pipeline 使用**连续片段**做 held-out validation。

这不是说所有工业语料都必须按“最后 10% token”切分；大型系统可能按 document、source、time、shard 等边界划分。

但对于当前单文件 Tiny Shakespeare，contiguous split 是最简单、清楚且不会把 token-level sequence 打散的方案。


## 9. Validation Data 能不能影响我们的决定？

Validation 的核心约束是：

> 不能被 optimizer 当作训练样本来更新参数。

但以后我们当然会查看 validation performance，判断：

- 模型是否只在 training data 上越来越好；
- held-out text 是否同步改善；
- 是否出现明显 overfitting trend。

所以 validation 不是“完全不能看”。

准确说法是：

```text
validation may inform evaluation / model-development decisions
but
validation examples are not optimizer training examples
```

如果不断围绕同一个 validation set 做大量人工调参，也可能逐渐对 validation set 本身产生选择偏差；正式研究通常还会保留独立 test set。

但本课程 Tiny Shakespeare pipeline 当前 roadmap 只要求 train / validation，因此本节不再增加第三个 split。


## 10. 16.2 Work Copy 的实际顺序

在新的 `68_..._work.ipynb` 中：

```text
1. 确认你已经在 Step 16.1 得到本地 cache
   data/tinyshakespeare/input.txt

2. 运行 local-cache bootstrap definition

3. 运行 bootstrap usage
   -> 得到 token_stream:(N,)
   -> 不联网

4. 运行 split helper definition

5. 运行 split usage/inspection
   -> train_tokens:(N_train,)
   -> val_tokens:(N_val,)

6. 检查两边长度相加仍为 N
```

做到这里就停止。

不要在本节继续写：

```text
block_size
get_batch
random start index
x/y shifted windows
```

这些属于 Step 16.3。


## 11. Step 16.2 材料边界

- 输入承接 Step 16.1：完整 character-level MLX token stream `token_stream:(N,)`。
- 新 work copy 使用 local-cache bootstrap helper 从 `data/tinyshakespeare/input.txt` 重建 tokenizer / `(N,)` stream；它不联网，cache 缺失时只提示先完成 16.1 acquisition。
- 当前采用 contiguous 90/10 split：`N_train=floor(0.9N)`，`N_val=N-N_train`。
- 输出仍是两个 1-D streams：`train_tokens:(N_train,)`、`val_tokens:(N_val,)`，并满足 `N_train+N_val=N`。
- held-out 的核心是 validation token positions 不参与模型参数更新；train / validation 原始 index ranges 不重叠。
- 后续 training windows 必须完全来自 train split，validation windows 必须完全来自 validation split，不能跨 split boundary；本节不实现 windows。
- 沿用 16.1 从完整 corpus 建立的 fixed character vocabulary；本节的数据隔离针对 sequence tokens / optimization examples，不重新设计 tokenizer。
- validation 可以用于未来 evaluation，但不能作为 optimizer training examples；本节不实现 loss / optimizer。
- 本轮只准备 Step 16.2，修正计数从 0/2 起；不进入 Step 16.3，不定义 `block_size`、batch sampler、shifted `(x,y)` batches，不运行 Notebook/测试/训练。
